# Rare Disasters

The model of [Wachter (2013)](https://doi.org/10.1111/jofi.12018), adapted from an example of EconPDEs.jl. Consumption grows smoothly most of the time, but at a rate `lambda` a disaster strikes and lowers it by a random factor `exp(Z)`, with `Z` negative. The state is the disaster intensity, which reverts to a mean:

```
d lambda = kappalambda (lambdabar - lambda) dt + nulambda sqrt(lambda) dB
```

Because disaster risk moves over time, so do risk premia and valuations, even though consumption is smooth outside disasters.

In [ ]:
Pollis.packages("EconPDEs", "Distributions", "Plots")

## From EconPDEs.jl

The code is the package's example, with Greek letters written in Latin (`mu`, `sigma`, `lambda`, `kappa`, `nu`, `rho`, `gamma`, `psi`, `phi`).

### The model

The unknown is the wealth-consumption ratio `p(lambda)`, with Epstein-Zin preferences. The jump size `Z` has its own distribution in the struct.

In [ ]:
using EconPDEs, Distributions, Plots

Base.@kwdef struct WachterModel{T<: Distribution}
    # consumption process parameters
    mu::Float64 = 0.025      # expected consumption growth
    sigma::Float64 = 0.02       # volatility of consumption growth

    # disaster process parameters
    lambdabar::Float64 = 0.0355  # long-run mean disaster intensity
    kappalambda::Float64 = 0.08      # mean-reversion speed of disaster intensity
    nulambda::Float64 = 0.067     # volatility of disaster intensity
    ZDistribution::T = Normal(-0.4, 0.25) # distribution of disaster jump size Z (log consumption drop); from Ian Martin higher order cumulants paper

    # utility parameters
    rho::Float64 = 0.012      # discount rate
    gamma::Float64 = 3.0        # relative risk aversion
    psi::Float64 = 1.1        # elasticity of intertemporal substitution
    phi::Float64 = 2.6        # leverage of dividends on consumption
end

In [ ]:
m = WachterModel()

### The grid and the guess

The intensity runs from zero to well above its long-run mean. The diffusion vanishes at `lambda = 0`, so no boundary condition is needed there.

In [ ]:
function initialize_stategrid(m::WachterModel; lambdan = 30)
  (; lambda = range(0.0, 0.1, length = lambdan))
end

In [ ]:
function initialize_guess(m::WachterModel, stategrid)
    lambdan = length(stategrid[:lambda])
    (; p = ones(lambdan))
end

stategrid = initialize_stategrid(m)
guess = initialize_guess(m, stategrid)

### The equation

The moment generating function `mgf` of `Z` carries the jump terms into the risk-free rate and into the pricing of the claim.

In [ ]:
function (m::WachterModel)(state::NamedTuple, u::NamedTuple)
    (; mu, sigma, lambdabar, kappalambda, nulambda, ZDistribution, rho, gamma, psi, phi) = m
    (; lambda) = state
    (; p, plambda_up, plambda_down, plambdalambda) = u

    # drift and volatility of lambda and p
    mulambda = kappalambda * (lambdabar - lambda)
    sigmalambda = nulambda * sqrt(lambda)
    # upwind on the sign of the drift of lambda
    plambda = (mulambda >= 0) ? plambda_up : plambda_down
    mup = plambda / p * mulambda + 0.5 * plambdalambda / p * sigmalambda^2
    sigmap_Zlambda = plambda / p * sigmalambda

    # market prices of risk
    kappa_Zc = gamma * sigma
    kappa_Zlambda = (gamma * psi - 1) / (psi - 1) * sigmap_Zlambda
    eta =  lambda * (mgf(ZDistribution, 1) - 1 + mgf(ZDistribution, -gamma) - 1 - (mgf(ZDistribution, 1 - gamma) - 1))

    # interest rate
    r = rho + mu / psi - (1 + 1 / psi) / 2 * gamma * sigma^2 - lambda * (mgf(ZDistribution, -gamma) - 1) + (1 / psi - gamma) / (1 - gamma) * lambda * (mgf(ZDistribution, 1 - gamma) - 1) - (gamma * psi - 1) / (2 * (psi - 1)) * sigmap_Zlambda^2

    # market pricing of the consumption claim
    pt = - p * (1 / p  + mu + mup + lambda * (mgf(ZDistribution, 1) - 1) - r - kappa_Zc * sigma - kappa_Zlambda * sigmap_Zlambda - eta)

    return (; pt)
end

### Solving it

In [ ]:
result = pdesolve(m, stategrid, guess)

### The solution

The wealth-consumption ratio falls as the disaster intensity rises: when a disaster is more likely, the claim is discounted more heavily.

In [ ]:
lambdas = stategrid[:lambda]
plot(lambdas, result.solution.p; xlabel = "disaster intensity lambda", ylabel = "wealth-consumption ratio p", legend = false)

## Source and Licence

The code under **From EconPDEs.jl** is adapted from `examples/asset_pricing/wachter.jl` of [EconPDEs.jl](https://github.com/matthieugomez/EconPDEs.jl), version 3.1.0, by Matthieu Gomez. Changes made in Pollis: Greek letters written in Latin.

The notice below covers the package's code only, not the parts added in Pollis.

The EconPDEs.jl package is licensed under the MIT "Expat" License:

> Copyright (c) 2015: Matthieu Gomez.
>
> Permission is hereby granted, free of charge, to any person obtaining
> a copy of this software and associated documentation files (the
> "Software"), to deal in the Software without restriction, including
> without limitation the rights to use, copy, modify, merge, publish,
> distribute, sublicense, and/or sell copies of the Software, and to
> permit persons to whom the Software is furnished to do so, subject to
> the following conditions:
>
> The above copyright notice and this permission notice shall be
> included in all copies or substantial portions of the Software.
>
> THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND,
> EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF
> MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT.
> IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY
> CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT,
> TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE
> SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.

## References

Wachter, J. A. (2013). Can time-varying risk of rare disasters explain aggregate stock market volatility? *The Journal of Finance* 68(3), 987–1035. [doi:10.1111/jofi.12018](https://doi.org/10.1111/jofi.12018)